In [ ]:
# v38b_gated_selector (offline/eval). Order: v38b (premises_FOL rows) -> v35 fallback (YNU) -> LoRA/parsefix.
# Abstain-safe: never overwrites a baseline answer with None. Analysis/eval pipeline only (NOT a live drop-in:
# live API has NL premises, no FOL).
print("v38b gated selector")

In [ ]:
import re
# ---------- v39-lite: canonical predicate ----------
def canon_atom(s):
    s=str(s).strip()
    s=re.sub(r'\(x\)|\(\s*x\s*\)','',s)
    s=s.strip()
    # FOL CamelCase atom -> as-is canonical key
    if re.fullmatch(r'[A-Za-z][A-Za-z0-9]*', s):
        return s
    # NL fallback: tokenize, drop stopwords/subjects, light de-inflect, join
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not'}
    toks=re.findall(r"[a-zA-Z]+", s.lower())
    out=[]
    for t in toks:
        if t in STOP: continue
        t=re.sub(r'(ies)$','y',t); t=re.sub(r'(es|s)$','',t); t=re.sub(r'(ing|ed)$','',t)
        out.append(t)
    return "_".join(out) if out else s.lower()

def _norm_tokens(text):
    text=re.sub(r'(?<!^)(?=[A-Z])',' ',str(text))
    toks=re.findall(r'[a-zA-Z]+', text.lower())
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not','one','least','according','premise',
          'premises','following','statement','true','based','above','can','be','inferred','supported','logically'}
    def _stem(t):
        if re.search(r'(ss|us|is)$', t): pass
        elif re.search(r'(ches|shes|xes|zes|ses)$', t): t=t[:-2]
        elif re.search(r'ies$', t): t=t[:-3]+'y'
        elif t.endswith('s'): t=t[:-1]
        t=re.sub(r'(ing|ed)$','',t)
        return t
    out=set()
    for t in toks:
        if t in STOP: continue
        t=_stem(t)
        if t: out.add(t)
    return out

In [ ]:
# ---------- FOL parser ----------
def parse_fol(fol):
    """Return ('rule',A,B) | ('uni',A) | ('uni_neg',A) | ('exist',A) | ('exist_neg',A) | None"""
    f=str(fol).replace('->','→').replace('¬','~').replace('∀','A').replace('∃','E')
    f=f.strip()
    # implication
    m=re.search(r'\(?\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*→\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*\)?', f)
    if m and f.startswith('A'):
        a=m.group(1).replace(' ',''); b=m.group(2).replace(' ','')
        an=a.startswith('~'); bn=b.startswith('~')
        return ('rule', (canon_atom(a.lstrip('~')),an), (canon_atom(b.lstrip('~')),bn))
    # quantified single atom
    m=re.search(r'^([AE])\s*x?\s*\(?\s*(~?)\s*([A-Za-z0-9]+)\s*\(x\)\s*\)?$', f)
    if m:
        quant,neg,pred=m.group(1),m.group(2)=='~',canon_atom(m.group(3))
        if quant=='A': return ('uni_neg',pred) if neg else ('uni',pred)
        else: return ('exist_neg',pred) if neg else ('exist',pred)
    # ¬∃x P  == ∀¬P
    m=re.search(r'~\s*E\s*x?\s*\(?\s*([A-Za-z0-9]+)\s*\(x\)', f)
    if m: return ('uni_neg',canon_atom(m.group(1)))
    return None

In [ ]:
# ---------- closure ----------
def build_closure(premises_fol):
    rules=[]; uni=set(); uni_neg=set(); exist=set()
    prov={}  # atom -> premise idx that introduced it (for path)
    for i,fol in enumerate(premises_fol):
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': rules.append((i,p[1],p[2]))
        elif p[0]=='uni': uni.add(p[1]); prov.setdefault(('pos',p[1]),[i])
        elif p[0]=='uni_neg': uni_neg.add(p[1]); prov.setdefault(('neg',p[1]),[i])
        elif p[0]=='exist': exist.add(p[1]); prov.setdefault(('ex',p[1]),[i])
    # forward positive: uni + (A->B, A pos, B pos-polarity) => B uni
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            # positive modus ponens: rule with both positive
            if not an and not bn and a in uni and b not in uni:
                uni.add(b); prov[('pos',b)]=prov.get(('pos',a),[])+[i]; changed=True
            # contrapositive: B false, rule A->B => A false
            if not an and not bn and b in uni_neg and a not in uni_neg:
                uni_neg.add(a); prov[('neg',a)]=prov.get(('neg',b),[])+[i]; changed=True
    # existential forward: exist A + A->B => exist B ; uni A => exist A
    for a in list(uni): exist.add(a); prov.setdefault(('ex',a),prov.get(('pos',a),[]))
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            if not an and not bn and a in exist and b not in exist:
                exist.add(b); prov[('ex',b)]=prov.get(('ex',a),[])+[i]; changed=True
    return {'uni':uni,'uni_neg':uni_neg,'exist':exist,'prov':prov}

In [ ]:
# ---------- query type + target ----------
def query_type(q):
    ql=str(q).lower()
    if re.search(r'\bat least one\b|\bsome\b|\bany\b|\bthere (is|exists)\b|does .* one', ql): return 'existential'
    if re.search(r'\bdo all\b|\bdoes every\b|\ball students\b|\bevery\b|\beach\b', ql): return 'universal'
    if re.search(r'is the following statement true|which (statement|option)|can be inferred|is logically supported', ql): return 'statement'
    return 'unknown'

def target_atom(q, atoms):
    qt=_norm_tokens(q)
    scored=[]
    for a in atoms:
        at=_norm_tokens(a)
        if not at: continue
        ov=len(qt & at)/len(at)   # fraction of atom tokens covered by question
        scored.append((ov,len(at & qt),a))
    scored.sort(reverse=True)
    if not scored: return None
    top=scored[0]
    if top[0] < 0.6 or top[1] < 1: return None
    # uniqueness: if a different atom ties on coverage AND raw overlap, ambiguous
    ties=[s for s in scored if abs(s[0]-top[0])<1e-9 and s[1]==top[1] and s[2]!=top[2]]
    if ties: return None
    return top[2]

In [ ]:
# ---------- projection with v35 convention + certificate ----------
def prove(premises_fol, question):
    cl=build_closure(premises_fol)
    atoms=cl['uni']|cl['uni_neg']|cl['exist']|{a for _,(a,_),(b,_) in [] }
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    qt=query_type(question); tgt=target_atom(question, allatoms)
    cert={'query_type':qt,'target':tgt,'positive':None,'negative':None,'answer':None,'premises_used':[],'abstain_reason':None}
    if tgt is None:
        cert['answer']=None; cert['abstain_reason']='target_not_matched'; return cert
    pos = tgt in cl['uni'] or tgt in cl['exist']
    neg = tgt in cl['uni_neg']
    cert['positive']=pos; cert['negative']=neg
    if qt=='existential':
        if neg:  # E1: forall-not target -> no instance (convention: wins even under positive conflict)
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='E1_universal_negative'
        elif pos:
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('ex',tgt),cl['prov'].get(('pos',tgt),[])); cert['proof_rule']='PE_witness'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    elif qt=='universal':
        if tgt in cl['uni']:  # PY: positive universal wins
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('pos',tgt),[]); cert['proof_rule']='PY_universal_positive'
        elif neg:
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='U1_universal_negative'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    else:
        cert['answer']=None; cert['abstain_reason']='statement_or_mc_out_of_scope'
    cert['premises_used']=sorted(set(cert['premises_used']))
    return cert

In [ ]:
def classify_option(opt):
    t=str(opt).strip(); tl=t.lower()
    if re.search(r"cannot be (determined|inferred)|undetermined|does not (support|allow)|no conclusion|insufficient", tl):
        return "META"
    # conditional / relative-clause distractor: "X who/that ... must/will/should ..." or "if ... then ..."
    if re.search(r"\bwho\b|\bthat\b", tl) and re.search(r"\b(must|will|should|then)\b", tl): return "CONDITIONAL"
    if re.search(r"^\s*if\b", tl): return "CONDITIONAL"
    if re.search(r"\bmust\b", tl): return "CONDITIONAL"   # malformed "must completes"
    if re.search(r"^\s*no\b", tl): return "UNIV_NEG"
    if re.search(r"^\s*(only some|some only)\b", tl): return "PARTIAL"
    if re.search(r"^\s*(at least one|some|there (is|exists))\b", tl): return "EXIST_POS"
    if re.search(r"^\s*(every|all|each)\b", tl): return "UNIV_POS"
    return "UNKNOWN_OPT"

def allatoms_of(fol):
    A=set()
    for f in fol:
        p=parse_fol(f)
        if not p: continue
        if p[0]=="rule": A.add(p[1][0]); A.add(p[2][0])
        else: A.add(p[1])
    return A

def eval_direct(kind, opt, cl, allatoms):
    atom=target_atom(opt, allatoms)
    if atom is None: return "UNSUPPORTED",None
    if kind=="UNIV_POS": return ("PROVEN" if atom in cl['uni'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="UNIV_NEG": return ("PROVEN" if atom in cl['uni_neg'] else ("DISPROVEN" if atom in cl['uni'] else "UNSUPPORTED")),atom
    if kind=="EXIST_POS": return ("PROVEN" if atom in cl['exist'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="PARTIAL":
        if atom in cl['exist'] and atom not in cl['uni'] and atom not in cl['uni_neg']: return "PROVEN",atom
        return ("DISPROVEN" if (atom in cl['uni'] or atom in cl['uni_neg']) else "UNSUPPORTED"),atom
    return "UNSUPPORTED",atom

def prove_mc_v38b(fol, options):
    cl=build_closure(fol); allatoms=allatoms_of(fol)
    labels=list("ABCD")[:len(options)]
    proven=[]; meta=None; prov_atom=None
    for lab,opt in zip(labels,options):
        k=classify_option(opt)
        if k=="META": meta=lab; continue
        if k in ("CONDITIONAL","UNKNOWN_OPT"): continue   # never selectable
        st,atom=eval_direct(k,opt,cl,allatoms)
        if st=="PROVEN": proven.append((lab,atom))
    cert={'answer':None,'rule':None,'premises_used':[],'abstain_reason':None}
    if len(proven)==1:
        lab,atom=proven[0]; cert['answer']=lab; cert['rule']='MC_unique_direct_proof'
        for key in [('pos',atom),('neg',atom),('ex',atom)]:
            if key in cl['prov']: cert['premises_used']=sorted(set(cl['prov'][key])); break
    elif len(proven)==0 and meta is not None:
        cert['answer']=meta; cert['rule']='MC_meta_cannot_determine'
    else:
        cert['abstain_reason']=('multiple_direct_proven' if proven else 'no_direct_and_no_meta')
    return cert
print('engine ready')

In [ ]:
# Optional v35 (original NL verifier) fallback for YNU rows where v38b abstains.
import os,sys,types,importlib.util,shutil
from pathlib import Path
TYPE1_PATCH=os.environ.get("TYPE1_PATCH","/kaggle/input/datasets/nguyenminhtric/exact-test/type1_fixed_patch/type1_fixed")
V35=None
try:
    P=Path(TYPE1_PATCH); W=Path("/kaggle/working/_v35rt"); shutil.rmtree(W,ignore_errors=True)
    (W/"app/type1_logic").mkdir(parents=True,exist_ok=True)
    for n in ["app/__init__.py","app/type1_logic/__init__.py"]:(W/n).write_text("")
    for n in ["parser.py","prompt.py","solver.py","verifier_v35.py"]: shutil.copy(P/n,W/"app/type1_logic"/n)
    am=types.ModuleType("app"); am.__path__=[str(W/"app")]; sys.modules["app"]=am
    tm=types.ModuleType("app.type1_logic"); tm.__path__=[str(W/"app/type1_logic")]; sys.modules["app.type1_logic"]=tm
    for n in ["parser","prompt"]:
        sp=importlib.util.spec_from_file_location(f"app.type1_logic.{n}",W/"app/type1_logic"/f"{n}.py"); m=importlib.util.module_from_spec(sp); sys.modules[f"app.type1_logic.{n}"]=m; sp.loader.exec_module(m)
    sp=importlib.util.spec_from_file_location("app.type1_logic.verifier_v35",W/"app/type1_logic/verifier_v35.py"); V35=importlib.util.module_from_spec(sp); sys.modules["app.type1_logic.verifier_v35"]=V35; sp.loader.exec_module(V35)
    print("v35 fallback loaded")
except Exception as e:
    print("v35 fallback NOT loaded (skipping):",type(e).__name__)

In [ ]:
# Gated selector
import re
def parse_opts(q): return [o[1].strip().replace("\n"," ") for o in re.findall(r"(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)", q, flags=re.S)]
def lora_answer(raw):
    m=re.findall(r"Final Answer:\s*([A-D]|Yes|No|Unknown|Uncertain)\b",raw or "",re.I)
    if not m: return "Unknown"
    v=m[0].strip(); v=v.upper() if re.fullmatch(r"[A-Da-d]",v) else v.title()
    return "Unknown" if v=="Uncertain" else v
def gated_select(row):
    fol=row.get("premises_FOL") or []; q=row.get("question",""); opts=row.get("options") or parse_opts(q)
    base=lora_answer(row.get("raw_output",""))
    # 1) v38b
    if fol:
        if row.get("is_mc") or (opts and not row.get("is_ynu")):
            c=prove_mc_v38b(fol,opts)
            if c.get("answer") is not None:
                return {"answer":c["answer"],"premises_used":c.get("premises_used",[]),"source":"v38b_mc","rule":c.get("rule")}
        else:
            c=prove(fol,q)
            if c.get("answer") is not None:
                return {"answer":c["answer"],"premises_used":c.get("premises_used",[]),"source":"v38b_ynu","rule":c.get("proof_rule")}
    # 2) v35 fallback (YNU only)
    if V35 is not None and not row.get("is_mc"):
        try:
            va,vp,vr=V35.verify(q,row.get("premises_NL") or [],base)
            if va is not None:
                return {"answer":va,"premises_used":vp or [],"source":"v35","rule":vr}
        except Exception: pass
    # 3) LoRA / parsefix baseline
    return {"answer":base,"premises_used":[],"source":"lora","rule":None}
print("gated_select ready")

In [ ]:
# Run selector on a full preds file (06b/06c). Reports baseline vs selected.
import json,glob,os
CANDS=["/kaggle/working/06c_generated_v4style_300_full_preds.json",
       "/kaggle/working/06b_generated_v4style_300_full_preds.json",
       "/kaggle/input/**/06*generated_v4style_300*full*preds.json",
       "/kaggle/input/**/06b_generated_v4style_300_smoke50_preds.json"]
def find():
    for c in CANDS:
        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])
        if h: return h[0]
    return None
PATH=find(); print("preds:",PATH)
rows=json.load(open(PATH)) if PATH else []
def norm_gold(g):
    g=str(g).strip(); return g.upper() if g.upper() in {"A","B","C","D"} else g.title()
b=s=n=0; ov=ovg=ovb=pu=0; src={}; sel_rows=[]
for r in rows:
    gold=norm_gold(r.get("gold")); base=lora_answer(r.get("raw_output",""))
    out=gated_select(r); a=out["answer"]; n+=1
    b+= (base==gold); s+= (a==gold); src[out["source"]]=src.get(out["source"],0)+1
    if out["source"].startswith("v38b") or out["source"]=="v35":
        ov+=1; pu+= (1 if out["premises_used"] else 0)
        if a!=base:
            if a==gold: ovg+=1
            elif base==gold: ovb+=1
    sr=dict(r); sr.update({"selected_answer":a,"selected_premises_used":out["premises_used"],"selected_source":out["source"]}); sel_rows.append(sr)
if rows:
    rep={"n":n,"baseline_acc":round(b/n,4),"selected_acc":round(s/n,4),"delta":round((s-b)/n,4),
         "overrides":ov,"override_good":ovg,"override_bad":ovb,"with_premises_used":pu,"source_dist":src}
    json.dump(rep,open("/kaggle/working/v38b_selector_summary.json","w"),indent=1)
    json.dump(sel_rows,open("/kaggle/working/v38b_selected_preds.json","w"),indent=1)
    print(f"baseline acc={rep['baseline_acc']}  selected acc={rep['selected_acc']}  (+{rep['delta']})")
    print(f"overrides={ov} good={ovg} bad={ovb} with_certificate={pu} | sources={src}")
    print("saved v38b_selector_summary.json + v38b_selected_preds.json")
else:
    print("No preds file found. Run 06b/06c full-gen first (needs LoRA raw_output per row), then re-run.")